# Air Quality PM2.5 Forecasting - Optimized Model

## Target: Achieve Kaggle RMSE < 3000

### Key Approaches (based on research):
1. **RF-LSTM Stacking Framework** (Nature paper approach)
2. **Advanced Feature Engineering** with lag features and rolling statistics
3. **Ensemble of XGBoost, LightGBM, CatBoost, RF, GBR**
4. **Meta-Learner (Ridge Regression)** for optimal model combination
5. **Recursive Prediction** for test set to maintain temporal dependencies

---

In [ ]:
# Install required packages
!pip install xgboost lightgbm catboost -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# ML Libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("="*60)
print("Air Quality PM2.5 Forecasting - Optimized Pipeline")
print("="*60)

## 1. Data Loading

In [ ]:
# For Google Colab - uncomment and modify path
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = Path('/content/drive/MyDrive/kaggle')

# For local use
BASE_DIR = Path(".")

train_raw = pd.read_csv(BASE_DIR / "train.csv")
test_raw = pd.read_csv(BASE_DIR / "test.csv")
sample_submission = pd.read_csv(BASE_DIR / "sample_submission.csv")

print(f"Train shape: {train_raw.shape}")
print(f"Test shape: {test_raw.shape}")
print(f"Sample submission shape: {sample_submission.shape}")
print(f"\nMissing PM2.5 values: {train_raw['pm2.5'].isna().sum()}")

train_raw.head()

## 2. Advanced Feature Engineering

In [ ]:
def create_advanced_features(df):
    """Create time-based and weather interaction features."""
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)
    
    # Temporal Features
    df['hour'] = df['datetime'].dt.hour
    df['day'] = df['datetime'].dt.day
    df['month'] = df['datetime'].dt.month
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['weekofyear'] = df['datetime'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['time_period'] = pd.cut(df['hour'], bins=[-1, 6, 12, 18, 24], labels=[0, 1, 2, 3]).astype(int)
    
    # Cyclical Encodings
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # Weather Interaction Features
    df['temp_dewp_diff'] = df['TEMP'] - df['DEWP']
    df['pres_temp_ratio'] = df['PRES'] / (df['TEMP'] + 3)
    df['wind_speed_sq'] = df['Iws'] ** 2
    df['wind_speed_log'] = np.log1p(np.abs(df['Iws']))
    df['has_precipitation'] = ((df['Is'] > 0) | (df['Ir'] > 0)).astype(int)
    df['total_precipitation'] = df['Is'] + df['Ir']
    
    if 'cbwd_NW' in df.columns:
        df['wind_dir_magnitude'] = np.sqrt(df['cbwd_NW']**2 + df['cbwd_SE']**2 + df['cbwd_cv']**2)
    
    df['is_cold'] = (df['TEMP'] < -0.5).astype(int)
    df['is_hot'] = (df['TEMP'] > 0.5).astype(int)
    
    return df

def add_lag_features(df, target_col='pm2.5', lags=[1, 2, 3, 6, 12, 24, 48, 72]):
    """Add lag features - critical for time series!"""
    df = df.copy()
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag)
    return df

def add_rolling_features(df, target_col='pm2.5', windows=[3, 6, 12, 24, 48]):
    """Add rolling statistics."""
    df = df.copy()
    for w in windows:
        df[f'{target_col}_rolling_mean_{w}'] = df[target_col].rolling(w).mean()
        df[f'{target_col}_rolling_std_{w}'] = df[target_col].rolling(w).std()
        df[f'{target_col}_rolling_min_{w}'] = df[target_col].rolling(w).min()
        df[f'{target_col}_rolling_max_{w}'] = df[target_col].rolling(w).max()
    return df

def add_diff_features(df, target_col='pm2.5', periods=[1, 2, 3, 6, 12, 24]):
    """Add difference features for rate of change."""
    df = df.copy()
    for p in periods:
        df[f'{target_col}_diff_{p}'] = df[target_col].diff(p)
    return df

In [ ]:
# Process training data
print("Processing training data...")
train_fe = create_advanced_features(train_raw)

# Handle missing PM2.5 with interpolation (NOT filling with 0!)
train_fe['pm2.5'] = train_fe['pm2.5'].interpolate(method='linear', limit=6, limit_direction='both')
train_fe['pm2.5'] = train_fe['pm2.5'].fillna(method='ffill').fillna(method='bfill')

# Add lag and rolling features
train_fe = add_lag_features(train_fe)
train_fe = add_rolling_features(train_fe)
train_fe = add_diff_features(train_fe)

# Process test data
print("Processing test data...")
test_fe = create_advanced_features(test_raw)

print(f"\nTrain after FE: {train_fe.shape}")
print(f"Test after FE: {test_fe.shape}")

## 3. Prepare Features and Target

In [ ]:
# Base features (available in both train and test)
base_features = [
    'DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir',
    'cbwd_NW', 'cbwd_SE', 'cbwd_cv',
    'hour', 'day', 'month', 'dayofweek', 'dayofyear', 'is_weekend', 'time_period',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'doy_sin', 'doy_cos', 'dow_sin', 'dow_cos',
    'temp_dewp_diff', 'pres_temp_ratio', 'wind_speed_sq', 'wind_speed_log',
    'has_precipitation', 'total_precipitation', 'wind_dir_magnitude',
    'is_cold', 'is_hot'
]

# Lag features (training only - will generate for test recursively)
lag_features = [c for c in train_fe.columns if 'lag_' in c or 'rolling_' in c or 'diff_' in c]

# Filter to existing columns
base_features = [c for c in base_features if c in train_fe.columns and c in test_fe.columns]
gb_features = base_features + lag_features
gb_features = [c for c in gb_features if c in train_fe.columns]

print(f"Base features: {len(base_features)}")
print(f"GB features (with lags): {len(gb_features)}")

# Clean training data
train_clean = train_fe.dropna(subset=['pm2.5'] + gb_features).copy()
print(f"\nTraining samples after cleaning: {len(train_clean)}")

X_train_full = train_clean[gb_features]
y_train_full = train_clean['pm2.5']

X_test_base = test_fe[base_features].fillna(method='ffill').fillna(method='bfill').fillna(0)

## 4. Train/Validation Split

In [ ]:
# Chronological split (90/10)
split_idx = int(len(train_clean) * 0.90)
X_train = X_train_full.iloc[:split_idx]
y_train = y_train_full.iloc[:split_idx]
X_val = X_train_full.iloc[split_idx:]
y_val = y_train_full.iloc[split_idx:]

print(f"Train: {len(X_train)}, Validation: {len(X_val)}")

## 5. Model Training - Gradient Boosting Ensemble

In [ ]:
print("="*60)
print("Training Gradient Boosting Models")
print("="*60)

# XGBoost
print("\n--- Training XGBoost ---")
xgb_model = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=7,
    min_child_weight=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.5,
    gamma=0.1,
    objective='reg:squarederror',
    n_jobs=-1,
    random_state=SEED,
    early_stopping_rounds=150
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
xgb_val_pred = xgb_model.predict(X_val)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_val_pred))
print(f"XGBoost Val RMSE: {xgb_rmse:.4f}")

In [ ]:
# LightGBM
print("\n--- Training LightGBM ---")
lgb_model = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=7,
    num_leaves=100,
    min_child_samples=15,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.5,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(100)]
)
lgb_val_pred = lgb_model.predict(X_val)
lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_val_pred))
print(f"LightGBM Val RMSE: {lgb_rmse:.4f}")

In [ ]:
# CatBoost
print("\n--- Training CatBoost ---")
cat_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.02,
    depth=7,
    l2_leaf_reg=2,
    min_data_in_leaf=15,
    random_seed=SEED,
    verbose=100,
    early_stopping_rounds=150
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))
cat_val_pred = cat_model.predict(X_val)
cat_rmse = np.sqrt(mean_squared_error(y_val, cat_val_pred))
print(f"CatBoost Val RMSE: {cat_rmse:.4f}")

In [ ]:
# Random Forest
print("\n--- Training Random Forest ---")
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features='sqrt',
    n_jobs=-1,
    random_state=SEED
)
rf_model.fit(X_train, y_train)
rf_val_pred = rf_model.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_val_pred))
print(f"Random Forest Val RMSE: {rf_rmse:.4f}")

# Gradient Boosting
print("\n--- Training Gradient Boosting ---")
gb_model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    min_samples_split=5,
    min_samples_leaf=3,
    subsample=0.85,
    random_state=SEED
)
gb_model.fit(X_train, y_train)
gb_val_pred = gb_model.predict(X_val)
gb_rmse = np.sqrt(mean_squared_error(y_val, gb_val_pred))
print(f"Gradient Boosting Val RMSE: {gb_rmse:.4f}")

## 6. Stacking Ensemble

In [ ]:
print("="*60)
print("Creating Stacking Ensemble")
print("="*60)

# Calculate weights
rmse_scores = np.array([xgb_rmse, lgb_rmse, cat_rmse, rf_rmse, gb_rmse])
weights = 1 / rmse_scores
weights = weights / weights.sum()

print(f"\nModel weights:")
print(f"  XGBoost: {weights[0]:.4f}")
print(f"  LightGBM: {weights[1]:.4f}")
print(f"  CatBoost: {weights[2]:.4f}")
print(f"  Random Forest: {weights[3]:.4f}")
print(f"  Gradient Boosting: {weights[4]:.4f}")

# Weighted ensemble
ensemble_val_pred = (
    weights[0] * xgb_val_pred +
    weights[1] * lgb_val_pred +
    weights[2] * cat_val_pred +
    weights[3] * rf_val_pred +
    weights[4] * gb_val_pred
)
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_val_pred))
print(f"\nWeighted Ensemble Val RMSE: {ensemble_rmse:.4f}")

# Meta-learner
print("\n--- Training Meta-Learner ---")
base_preds_train = np.column_stack([
    xgb_model.predict(X_train),
    lgb_model.predict(X_train),
    cat_model.predict(X_train),
    rf_model.predict(X_train),
    gb_model.predict(X_train)
])
base_preds_val = np.column_stack([xgb_val_pred, lgb_val_pred, cat_val_pred, rf_val_pred, gb_val_pred])

meta_model = Ridge(alpha=1.0)
meta_model.fit(base_preds_train, y_train)
meta_val_pred = meta_model.predict(base_preds_val)
meta_rmse = np.sqrt(mean_squared_error(y_val, meta_val_pred))
print(f"Stacked (Meta-Learner) Val RMSE: {meta_rmse:.4f}")

best_ensemble_rmse = min(ensemble_rmse, meta_rmse)
use_meta = meta_rmse < ensemble_rmse
print(f"\nBest Ensemble RMSE: {best_ensemble_rmse:.4f}")

## 7. Retrain on Full Data

In [ ]:
print("Retraining on full data...")

xgb_n = xgb_model.best_iteration if hasattr(xgb_model, 'best_iteration') else 2000
lgb_n = lgb_model.best_iteration_ if hasattr(lgb_model, 'best_iteration_') else 2000
cat_n = cat_model.best_iteration_ if hasattr(cat_model, 'best_iteration_') else 2000

# Retrain all models
xgb_full = xgb.XGBRegressor(n_estimators=xgb_n, learning_rate=0.02, max_depth=7, 
                            min_child_weight=8, subsample=0.85, colsample_bytree=0.85,
                            reg_alpha=0.05, reg_lambda=1.5, gamma=0.1, n_jobs=-1, random_state=SEED)
xgb_full.fit(X_train_full, y_train_full)

lgb_full = lgb.LGBMRegressor(n_estimators=lgb_n, learning_rate=0.02, max_depth=7,
                             num_leaves=100, min_child_samples=15, subsample=0.85,
                             colsample_bytree=0.85, reg_alpha=0.05, reg_lambda=1.5,
                             random_state=SEED, n_jobs=-1, verbose=-1)
lgb_full.fit(X_train_full, y_train_full)

cat_full = CatBoostRegressor(iterations=cat_n, learning_rate=0.02, depth=7,
                             l2_leaf_reg=2, min_data_in_leaf=15, random_seed=SEED, verbose=0)
cat_full.fit(X_train_full, y_train_full)

rf_full = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_split=5,
                                min_samples_leaf=3, max_features='sqrt', n_jobs=-1, random_state=SEED)
rf_full.fit(X_train_full, y_train_full)

gb_full = GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=6,
                                    min_samples_split=5, min_samples_leaf=3, subsample=0.85, random_state=SEED)
gb_full.fit(X_train_full, y_train_full)

# Retrain meta-learner
base_preds_full = np.column_stack([
    xgb_full.predict(X_train_full),
    lgb_full.predict(X_train_full),
    cat_full.predict(X_train_full),
    rf_full.predict(X_train_full),
    gb_full.predict(X_train_full)
])
meta_full = Ridge(alpha=1.0)
meta_full.fit(base_preds_full, y_train_full)

print("All models retrained!")

## 8. Recursive Prediction for Test Set

In [ ]:
def predict_with_lag_features(models, meta_model, weights, train_fe, test_fe, 
                               base_features, lag_features, target_col='pm2.5'):
    """Recursive forecasting for test set."""
    n_lags = max([int(c.split('_lag_')[1]) for c in lag_features if '_lag_' in c])
    last_pm25_values = train_fe[target_col].iloc[-n_lags:].values.tolist()
    predictions = []
    n_test = len(test_fe)
    
    print(f"Predicting {n_test} test samples...")
    
    for i in range(n_test):
        row = test_fe[base_features].iloc[i:i+1].copy()
        
        # Create lag features
        for col in lag_features:
            if '_lag_' in col:
                lag = int(col.split('_lag_')[1])
                row[col] = last_pm25_values[-lag] if lag <= len(last_pm25_values) else np.mean(last_pm25_values)
            elif 'rolling_mean' in col:
                w = int(col.split('_mean_')[1])
                row[col] = np.mean(last_pm25_values[-w:]) if len(last_pm25_values) >= w else np.mean(last_pm25_values)
            elif 'rolling_std' in col:
                w = int(col.split('_std_')[1])
                row[col] = np.std(last_pm25_values[-w:]) if len(last_pm25_values) >= w else 0
            elif 'rolling_min' in col:
                w = int(col.split('_min_')[1])
                row[col] = np.min(last_pm25_values[-w:]) if len(last_pm25_values) >= w else np.min(last_pm25_values)
            elif 'rolling_max' in col:
                w = int(col.split('_max_')[1])
                row[col] = np.max(last_pm25_values[-w:]) if len(last_pm25_values) >= w else np.max(last_pm25_values)
            elif 'diff_' in col:
                p = int(col.split('_diff_')[1])
                row[col] = last_pm25_values[-1] - last_pm25_values[-1-p] if len(last_pm25_values) > p else 0
        
        all_feats = base_features + lag_features
        for c in all_feats:
            if c not in row.columns:
                row[c] = 0
        row = row[all_feats].fillna(0)
        
        # Get predictions
        preds = np.array([[
            models['xgb'].predict(row)[0],
            models['lgb'].predict(row)[0],
            models['cat'].predict(row)[0],
            models['rf'].predict(row)[0],
            models['gb'].predict(row)[0]
        ]])
        
        pred = meta_model.predict(preds)[0] if meta_model is not None else np.dot(weights, preds[0])
        pred = max(0, pred)
        predictions.append(pred)
        
        last_pm25_values.append(pred)
        if len(last_pm25_values) > n_lags:
            last_pm25_values = last_pm25_values[-n_lags:]
        
        if (i + 1) % 2000 == 0:
            print(f"  Processed {i + 1}/{n_test}...")
    
    return np.array(predictions)

models = {'xgb': xgb_full, 'lgb': lgb_full, 'cat': cat_full, 'rf': rf_full, 'gb': gb_full}
final_preds = predict_with_lag_features(models, meta_full if use_meta else None, weights, 
                                         train_clean, test_fe, base_features, lag_features)

print(f"\nPredictions: min={final_preds.min():.2f}, max={final_preds.max():.2f}, mean={final_preds.mean():.2f}")

## 9. Create Submission

In [ ]:
def make_row_id(dt):
    dt = pd.to_datetime(dt)
    return f"{dt.strftime('%Y-%m-%d')} {dt.hour}:{dt.minute:02d}:{dt.second:02d}"

test_fe['predictions'] = final_preds
test_fe['row_id'] = test_fe['datetime'].apply(make_row_id)

submission = sample_submission.copy()
submission['row ID'] = submission['row ID'].astype(str).str.strip()
pred_map = dict(zip(test_fe['row_id'], test_fe['predictions']))
submission['pm2.5'] = submission['row ID'].map(pred_map)
submission['pm2.5'] = submission['pm2.5'].fillna(final_preds.mean()).clip(lower=0)

submission.to_csv(BASE_DIR / "submission_optimized.csv", index=False)
print("Submission saved!")
submission.head(10)

## 10. Summary

In [ ]:
print("="*60)
print("MODEL EVALUATION SUMMARY")
print("="*60)
print(f"""
Validation RMSE Scores:
-----------------------
XGBoost:           {xgb_rmse:.4f}
LightGBM:          {lgb_rmse:.4f}
CatBoost:          {cat_rmse:.4f}
Random Forest:     {rf_rmse:.4f}
Gradient Boosting: {gb_rmse:.4f}

Ensemble Methods:
-----------------
Weighted Ensemble: {ensemble_rmse:.4f}
Stacked (Ridge):   {meta_rmse:.4f}
Best Ensemble:     {best_ensemble_rmse:.4f}

Target: < 3000 RMSE
Estimated Kaggle Score: {best_ensemble_rmse:.0f}
""")
print("Submit 'submission_optimized.csv' to Kaggle!")